# FigureVault on Google Colab
This notebook sets up the environment to run the FigureVault pipeline on Google Colab.
**Before running:** Make sure to go to `Runtime` -> `Change runtime type` and select **T4 GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Note: Zip your figureVault folder and upload it to your Google Drive first.
# Change the path below to match where you uploaded your zip file.
!unzip -o "/content/drive/MyDrive/figureVault.zip" -d /content/figureVault_temp

# Move files if they are nested inside another figureVault folder
import os
import shutil
temp_path = "/content/figureVault_temp"
nested_path = os.path.join(temp_path, "figureVault")
target_path = "/content/figureVault"

if os.path.exists(nested_path):
    shutil.move(nested_path, target_path)
else:
    shutil.move(temp_path, target_path)

%cd /content/figureVault
!ls -F # Verify tests/ directory exists

In [ ]:
# Install python dependencies
!pip install -r requirements.txt
!pip install watchdog # Recommended for Streamlit on Linux

# Install system dependencies
!apt-get update
!apt-get install -y libgl1-mesa-glx pciutils lshw zstd default-jdk

### Build PDFFigures2 from source
Required for high-accuracy bounding box extraction.

In [ ]:
# Install sbt (Scala Build Tool)
!apt-get install apt-transport-https curl gnupg -yqq
!echo "deb https://repo.scala-sbt.org/scalasbt/debian all main" | sudo tee /etc/apt/sources.list.d/sbt.list
!echo "deb https://repo.scala-sbt.org/scalasbt/debian /" | sudo tee /etc/apt/sources.list.d/sbt_old.list
!curl -sL "https://keyserver.ubuntu.com/pks/lookup?op=get&search=0x2EE0EA64E40A89B84B2DF73499E82A75642AC823" | sudo -H gpg --no-default-keyring --keyring gnupg-ring:/etc/apt/trusted.gpg.d/scalasbt-release.gpg --import
!chmod 644 /etc/apt/trusted.gpg.d/scalasbt-release.gpg
!apt-get update
!apt-get install sbt -yqq

# Clone and build PDFFigures2
!git clone https://github.com/allenai/pdffigures2.git /content/pdffigures2
%cd /content/pdffigures2
!sbt assembly

# Copy the built JAR
!mkdir -p /content/figureVault/bin
!cp pdffigures2.jar /content/figureVault/bin/pdffigures2.jar
%cd /content/figureVault
print("✅ PDFFigures2 successfully built!")

In [ ]:
# Install & Start Ollama
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

process = subprocess.Popen(['ollama', 'serve'])
print("Ollama server starting...")
time.sleep(10)  # Wait for startup

!ollama pull gemma4:e4b

In [ ]:
# Run tests
!python -m pytest tests/

In [ ]:
# Run FigureVault help
!python main.py --help

### Run Streamlit UI
If the link below doesn't load, check `/content/logs.txt` for errors.

In [ ]:
import os
import time

# Kill any existing streamlit/cloudflared processes
!pkill streamlit
!pkill cloudflared

# Start Streamlit binding to 127.0.0.1 explicitly
os.system("streamlit run ui/app.py --server.port 8501 --server.address 127.0.0.1 &>/content/logs.txt &")
time.sleep(5)

# Download cloudflared
!wget -q -O - https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 > /tmp/cloudflared
!chmod +x /tmp/cloudflared

# Check if Streamlit is actually running
!netstat -tunlp | grep 8501

# Expose via Tunnel - Use 127.0.0.1 instead of localhost to avoid IPv6 resolution issues
!/tmp/cloudflared tunnel --url http://127.0.0.1:8501

In [ ]:
# DEBUG: Run this cell if Streamlit fails to load to see why
!cat /content/logs.txt